# 4 — Voronoi analysis (area per lipid, neighbours)

All-atom CHARMM36 binary bilayer (DOPC/DOPA, DOPC/DOPG, DOPC/DLPG, DOPC/DLPA).

Periodic 2D Voronoi tessellation of phosphates per leaflet → area per lipid (APL), mean neighbour count, and like-neighbour fraction (mixing signal). Single frame gives one snapshot APL; a trajectory gives a distribution.


## Setup

In [ ]:
# !pip install "MDAnalysis>=2.4" numpy scipy matplotlib
%matplotlib inline
import os, re, itertools
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

## Config — edit these

You have **no trajectory**, so `TRAJ = None` loads the single `.gro`
frame (a **snapshot**). When you have an `.xtc`/`.trr`, set `TRAJ` to it
and every cell below averages over frames automatically.

In [ ]:
TOPOLOGY = "md.tpr"        # topology (has bonds + masses)
STRUCTURE= "md.gro"        # single-frame structure (.gro or .pdb)
TRAJ     = None            # e.g. "md_center.xtc" when you have one; None -> use STRUCTURE

LIPIDS   = ["DOPC", "DOPA"]        # the TWO lipids in THIS system
HEAD     = "P"                     # headgroup / leaflet marker
STEP     = 1                       # frame stride (only matters with a trajectory)
OUTDIR   = "results"; os.makedirs(OUTDIR, exist_ok=True)

## Load

In [ ]:
u = mda.Universe(TOPOLOGY, TRAJ) if TRAJ else mda.Universe(TOPOLOGY, STRUCTURE)
has_bonds = len(getattr(u, "bonds", [])) > 0
present   = [r for r in LIPIDS if len(u.select_atoms(f"resname {r}")) > 0]
nframes   = len(u.trajectory)
print(f"{len(u.atoms)} atoms, {nframes} frame(s), bonds={has_bonds}")
print("Lipids present:", present)
if nframes == 1:
    print("NOTE: single frame -> snapshot only (no time averaging).")

## Voronoi per leaflet

In [ ]:
from scipy.spatial import Voronoi
def poly_area(p): x,y=p[:,0],p[:,1]; return 0.5*abs(np.dot(x,np.roll(y,1))-np.dot(y,np.roll(x,1)))
def tile(pts,Lx,Ly):
    sh=[(i,j) for i in (-1,0,1) for j in (-1,0,1)]
    return (np.vstack([pts+np.array([a*Lx,b*Ly]) for a,b in sh]),
            np.concatenate([np.arange(len(pts)) for _ in sh]))
def leaflet_voronoi(pts,rt,Lx,Ly):
    N=len(pts); tp,tmap=tile(pts,Lx,Ly); vor=Voronoi(tp); c=np.arange(4*N,5*N)
    ar=np.full(N,np.nan)
    for k,pi in enumerate(c):
        reg=vor.regions[vor.point_region[pi]]
        if reg and -1 not in reg: ar[k]=poly_area(vor.vertices[reg])
    neigh=[set() for _ in range(N)]
    for i,j in vor.ridge_points:
        if 4*N<=i<5*N: neigh[i-4*N].add(tmap[j])
        if 4*N<=j<5*N: neigh[j-4*N].add(tmap[i])
    for k in range(N): neigh[k].discard(k)
    nn=np.array([len(neigh[k]) for k in range(N)],float)
    lk=np.array([np.mean([rt[m]==rt[k] for m in neigh[k]]) if neigh[k] else np.nan for k in range(N)])
    return ar,nn,lk

P=u.select_atoms(f"name {HEAD} and resname "+" ".join(present))
rt_all=np.array([a.resname for a in P])
acc={r:{"apl":[],"nn":[],"like":[]} for r in present}
for ts in u.trajectory[::STEP]:
    Lx,Ly=ts.dimensions[:2]; pos=P.positions.copy(); pos[:,0]%=Lx; pos[:,1]%=Ly
    zmid=pos[:,2].mean()
    for mask in (pos[:,2]>=zmid,pos[:,2]<zmid):
        sub=pos[mask][:,:2]; rs=rt_all[mask]
        if len(sub)<4: continue
        ar,nn,lk=leaflet_voronoi(sub,rs,Lx,Ly)
        for r in present:
            m=rs==r
            acc[r]["apl"]+=list(ar[m][~np.isnan(ar[m])]); acc[r]["nn"]+=list(nn[m])
            acc[r]["like"]+=list(lk[m][~np.isnan(lk[m])])
print(f"{'lipid':<8}{'APL(A^2)':>10}{'SEM':>8}{'<neigh>':>10}{'like':>8}")
for r in present:
    apl=np.array(acc[r]["apl"]); nn=np.array(acc[r]["nn"]); lk=np.array(acc[r]["like"])
    if apl.size:
        print(f"{r:<8}{apl.mean():10.2f}{apl.std()/np.sqrt(apl.size):8.2f}{nn.mean():10.2f}{lk.mean():8.3f}")
print("box-estimate APL = 2*Lx*Ly/Nlipids =", round(2*Lx*Ly/len(P.residues),2), "A^2")